In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
HERE = %pwd
sys.path.append(os.path.dirname(HERE))

%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
    
import numpy as np
import pandas as pd
import copy
import pickle
import time
import collections
from tqdm import tqdm
from collections import defaultdict

In [ ]:
from src import utils
rng = utils.set_seed()

dir_parent = utils.dir_parent
version_exp = utils.version_exp
dir_workspace = f"{dir_parent}/research/TFCSR"

In this notebook, for ablation study, we only changed
- candidate_size = 50 (default option was 10)
- at_K = 10 (default option was 5)

In [ ]:
def compute(dict_data, llmreranker, dir_llmreranker):
    # flag
    dict_flag = dict_data["flag"]
    
    # queries
    dict_text = dict_data["profile"]
    dict_text.update(dict_data["concat"])
    
    # documents
    d_documents = dict_data["items"]["candidates"]
    
    # compute reranking score
    for text_type, d_query in dict_text.items():
        t = text_type.replace("concat_", "").replace("separate_", "")
        d_flag = dict_flag[t]
        ner = dict_data["ner"]
        
        dir_res = f"{dir_llmreranker}/{text_type}_{utils.rename(llmreranker.llm.model_name)}_instdefault_{ner}"
        os.makedirs(dir_res, exist_ok=True)   

        ddict_res = llmreranker.load(dir_res, d_query, d_flag, d_documents)

        df = pd.concat([pd.DataFrame(d["log"]) for d in ddict_res.values()], axis=1).T
        s_log = llmreranker.llm.compute_log(df_log=df)
        print(text_type)
        display(s_log)


def run(llmreranker, data_name, N_icl=[1,3,5], flag_replace_NER=False, candidate_size=10, at_K=5):
    from src.data_loader import Loader
    loader = Loader(dir_workspace, version_exp, data_name, N_icl=N_icl, flag_replace_NER=flag_replace_NER)
    dict_data = loader.load_data()

    dir_llmreranker = f"{dir_workspace}/LLMreranking_data/{version_exp}/{data_name}/candidate{candidate_size}_atK{at_K}"
    os.makedirs(dir_llmreranker, exist_ok=True)    

    print(f"{utils.rename(llmreranker.llm.model_name):30} {data_name:30} NER{flag_replace_NER} candidate{candidate_size} @K{at_K}")
    compute(dict_data, llmreranker, dir_llmreranker)

In [ ]:
data_names = ["MovieLens", "Job"] + [f"ARD_{a}" for a in [
    "CDs_and_Vinyl", "Movies_and_TV", "Toys_and_Games", "Sports_and_Outdoors"
]]

flag_replace_NER = [False, True][0]
N_icl = [1]

candidate_size = 50
at_K = 10
from src.reranker_llm import LLMReranker
llmreranker = LLMReranker(candidate_size=candidate_size)   

model_names_llm = [
    "gpt-5.1-2025-11-13_reasoning_none",
    "gpt-5.4-2026-03-05_reasoning_none",
    "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
]
for model_name_llm in model_names_llm:
    llm = utils.load_llm(model_name=model_name_llm)
    llmreranker.fit(llm)
    for data_name in data_names:
        run(llmreranker, data_name, N_icl=N_icl, flag_replace_NER=flag_replace_NER, candidate_size=candidate_size, at_K=at_K)